In [1]:
# Transformers example -> Sorting a sequence of numbers


In [1]:
import haiku as hk
import jax
import jax.numpy as jnp
import jax.random as jrandom
import optax

from probjax.nn.helpers import LearnedPosEmbed
from probjax.nn.transformers import Transformer

In [2]:
VOCAB_SIZE = 10

In [3]:
def generate_data(key, n, T, vocab_size=10):
    sequences = jrandom.randint(key, (n, T,1), 0, vocab_size, dtype=jnp.int32)

    sequences_sorted = jnp.sort(sequences, axis=-2)

    return sequences, sequences_sorted

inputs, labels = generate_data(jax.random.PRNGKey(0), 1000, 10, VOCAB_SIZE)

I0000 00:00:1701021795.437777   86584 tfrt_cpu_pjrt_client.cc:349] TfrtCpuClient created.
2023-11-26 19:03:17.432360: W external/xla/xla/service/gpu/nvptx_compiler.cc:708] The NVIDIA driver's CUDA version is 11.7 which is older than the ptxas CUDA version (11.8.89). Because the driver is older than the ptxas version, XLA is disabling parallel compilation, which may slow down compilation. You should update your NVIDIA driver or use the NVIDIA-provided CUDA forward compatibility packages.


In [4]:
key = jrandom.PRNGKey(0)

In [5]:

@hk.transform
def f(x):
    # Embed
    dim = 10
    x = jnp.squeeze(hk.Embed(VOCAB_SIZE, dim)(x), -2)
    # x = jnp.squeeze(jax.nn.one_hot(x, VOCAB_SIZE))
    x = LearnedPosEmbed(max_seq_len=100)(x)
    # Encode
    model = Transformer(num_heads=1, num_layers=2, attn_size=x.shape[-1], widening_factor=10)
    embedding = model(x, context=None)
    logits = hk.Linear(VOCAB_SIZE)(embedding)
    return logits


In [6]:
params = f.init(key, inputs)
outputs = f.apply(params, key, inputs)

In [7]:
optimizer = optax.adam(1e-3)
opt_state = optimizer.init(params)

In [8]:

def loss_fn(params, inputs, outputs, rng):
    inp_data, labels = inputs, outputs
    logits = f.apply(params, rng, inp_data)
    labels = jnp.squeeze(jax.nn.one_hot(labels, VOCAB_SIZE), -2)
    loss = optax.softmax_cross_entropy(logits, labels).mean()
    return loss

def acc(params, inputs, outputs, rng):
    inp_data, labels = inputs, outputs
    logits = f.apply(params, rng, inp_data)
    labels = jnp.squeeze(jax.nn.one_hot(labels, VOCAB_SIZE), -2)
    acc = (logits.argmax(axis=-1) == labels.argmax(-1)).mean()
    return acc

@jax.jit
def update(params, inputs, outputs, rng, opt_state):
    loss, grads = jax.value_and_grad(loss_fn)(params, inputs, outputs, rng)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    return loss, params, opt_state

In [9]:
train_seq_len = [5,10,100]
for i in range(20000):
    key, subkey, key2 = jrandom.split(jrandom.PRNGKey(i), 3)
    inputs, labels = generate_data(key, 512, train_seq_len[i%len(train_seq_len)], vocab_size=VOCAB_SIZE)
    loss, params, opt_state = update(params, inputs, labels, subkey, opt_state)
    if (i % 1000) == 0:
        accuracy = acc(params, inputs, labels, key2)
        print(accuracy, loss)

0.08359375 2.6814287
0.8257813 0.44563124
0.8914844 0.2556495
0.98164064 0.060683828
0.95585936 0.12270029
0.9382031 0.20378906
0.9964844 0.018474406
0.9873047 0.045729276
0.9483594 0.15808557
0.99726564 0.0069821454
0.97031254 0.062354214
0.9608984 0.10438451
0.99921876 0.005149496
0.9927735 0.027144969
0.95964843 0.24229448
0.9996094 0.0036426985
0.99121094 0.018832121
0.9731836 0.084530294
0.9996094 0.0027213735
0.9966797 0.016526243


In [10]:
input = jax.random.randint(key, (1, 4,1),0, 10,dtype=jnp.int32)
outputs = f.apply(params, key + 2, input)
print(input[0,...,0])
print(outputs.argmax(-1)[0])

[0 6 6 8]
[0 6 6 8]


In [12]:
input = jax.random.randint(key, (1, 10,1),0, 10,dtype=jnp.int32)
outputs = f.apply(params, key + 2, input)
print(input[0,...,0])
print(outputs.argmax(-1)[0])

[0 0 2 1 9 1 3 0 6 5]
[0 0 0 1 1 2 3 5 6 9]
